<!-- Werner: copy every cell BELOW this one into IEAP-Python-Series04-pre.ipynb, replacing the 'To be completed by BINCHE Werner' cell. Then delete this cell. -->

**Cells for Parts 3 and 4 (BINCHE Werner).** They use `signal`, `time`, `phases` and `derivative_central` defined in Parts 1–2, so they only run inside the shared notebook.

## 3. Estimate the tangential speed of the signal

### 3.1 Tangential velocity vs. tangential speed

For a point whose position is $\vec{r}(t)$:

- The **(tangential) velocity** is a **vector**: $\vec{v}(t) = \dfrac{d\vec{r}}{dt}$. It has a magnitude *and* a direction,
  and this direction is always tangent to the trajectory (hence "tangential"). In 3D it has 3 components $(\dot x, \dot y, \dot z)$.
- The **(tangential) speed** is a **scalar**: the magnitude (Euclidean norm) of the velocity vector,
  $v(t) = \lVert\vec{v}(t)\rVert \ge 0$. It is the rate at which the distance travelled along the path (arc length $s$) increases: $v = ds/dt$.
  It has no direction and is never negative.

Example: in uniform circular motion the speed is constant, but the velocity changes all the time (its direction rotates).

**References:** Wikipedia, *Velocity* — https://en.wikipedia.org/wiki/Velocity ;
Wikipedia, *Speed* — https://en.wikipedia.org/wiki/Speed ;
Wikipedia, *Arc length* — https://en.wikipedia.org/wiki/Arc_length

### 3.2 Tangential speed of a 3D movement

$$
v(t) = \lVert\vec{v}(t)\rVert = \sqrt{\dot{x}(t)^2 + \dot{y}(t)^2 + \dot{z}(t)^2}
$$

### 3.3 Tangential speed of an nD movement

For a position $\vec{r}(t) = \big(r_1(t), \dots, r_n(t)\big) \in \mathbb{R}^n$:

$$
v(t) = \lVert\dot{\vec{r}}(t)\rVert_2 = \sqrt{\sum_{k=1}^{n} \dot{r}_k(t)^2}
$$

In 1D this reduces to $v = |\dot{x}|$. For our signal:
$v(t) = 2\pi\sqrt{\cos^2(2\pi t) + \cos^2(2\pi t + \frac{\pi}{4}) + \cos^2(2\pi t + \frac{\pi}{2})}$, which is not constant
(the trajectory is an ellipse, not a circle traversed uniformly).

### 3.4 Function design

- **Algorithm:** (1) estimate the velocity vector with the central difference (more accurate, no time shift),
  (2) take the Euclidean norm of each row (`np.linalg.norm(..., axis=1)`).
- **Input order: `(signal, time)`**, for the same reasons as the derivative functions (quantity first, variable of
  differentiation second; consistent with `derivative_central(signal, time)`, which it calls).
- **Output shape:** `(n_samples, 1)` — one scalar speed per sample, kept as a column vector, consistent with `time`
  so it can be plotted directly against it. It works for any number of dimensions `d` (1D, 2D, 3D, … nD).

In [ ]:
# References: https://en.wikipedia.org/wiki/Speed , https://en.wikipedia.org/wiki/Velocity


def tangential_speed(signal, time):
    # Tangential speed = Euclidean norm of the velocity vector, for an nD movement.
    #   v(t) = sqrt( sum_k (dr_k/dt)^2 )
    # Inputs : signal (n, d) samples over rows, dims over columns (d = 1, 2, 3, ...) ;
    #          time (n,) or (n, 1)
    # Returns: numpy array of shape (n, 1), one speed value per sample.
    velocity = derivative_central(signal, time)          # (n, d)
    return np.linalg.norm(velocity, axis=1, keepdims=True)  # (n, 1)


speed = tangential_speed(signal, time)
print(f"signal.shape = {signal.shape} -> speed.shape = {speed.shape}")
print(f"speed: min = {speed.min():.3f}, max = {speed.max():.3f}")

### 3.5 Test cell

For each test case the expected speed is first derived analytically, then compared to the function output with `assert`.
A tolerance is needed because finite differences are an approximation (relative error of the order of $(\omega h)^2$ here).

| Case | Movement $\vec r(t)$ | Velocity $\dot{\vec r}(t)$ | Expected speed $v(t)$ |
|---|---|---|---|
| 1D linear | $x = a t + b$ | $\dot x = a$ | $\lvert a\rvert$ (constant) |
| 1D sine | $x = A\sin(\omega t)$ | $\dot x = A\omega\cos(\omega t)$ | $A\omega\,\lvert\cos(\omega t)\rvert$ |
| 2D circle | $(R\cos\omega t,\ R\sin\omega t)$ | $(-R\omega\sin\omega t,\ R\omega\cos\omega t)$ | $R\omega$ (constant) |
| 3D helix | $(R\cos\omega t,\ R\sin\omega t,\ c\,t)$ | $(-R\omega\sin\omega t,\ R\omega\cos\omega t,\ c)$ | $\sqrt{R^2\omega^2 + c^2}$ |
| 3D our signal | $\sin(2\pi t + k\Delta\varphi)$, $k=0,1,2$ | $2\pi\cos(2\pi t + k\Delta\varphi)$ | $2\pi\sqrt{\sum_k \cos^2(2\pi t + k\Delta\varphi)}$ |
| nD linear | $\vec r = \vec a\, t + \vec b$, $\vec a\in\mathbb{R}^n$ | $\vec a$ | $\lVert\vec a\rVert$ (e.g. $\vec a = (1,\dots,1)\Rightarrow\sqrt{n}$) |

For linear movements finite differences are **exact** (the second derivative is zero), so the tolerance can be very tight.
For the 1D sine, $\lvert\cos\rvert$ has kinks at its zeros, so the comparison uses an absolute tolerance.